<a href="https://colab.research.google.com/github/Priyanshujindal/Speak-with-the-PDF-Mini-App-RAG-/blob/main/Module_3_Assignment_OpenRouter_completed_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Please run this cell, to install the dependencies
! pip install pypdf openai pinecone


# Speak with PDF Assignment

**Assignment type:** RAG mini-app with three required functions  
**File to submit:** `pdf_ai.ipynb` (the notebook produced from this spec)  
**Language:** Python 3.x (Codio / Jupyter)  
**Time estimate:** 2–3 hours

---

## Objective

Build a **“Speak with PDF” RAG pipeline** that runs inside a Jupyter notebook and is compatible with **NBGrader**. You will implement **three functions**:

1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> List[str]`  
2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`  
3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Grading will use OpenAI keys.** Learners must provide valid `OPENAI_API_KEY` (and Pinecone credentials if using Pinecone).

---

## What you will build

A tiny RAG system that:
- loads a PDF,
- extracts and **chunks** text,
- builds a **vector index** from OpenAI embeddings (Pinecone recommended; in‑memory/FAISS acceptable),
- answers a natural-language query by retrieving top‑k relevant chunks and generating an answer with an OpenAI chat model.

This mirrors production flow (PDF → chunks → embeddings → index → retrieval → answer).


---

## Tools & Libraries

- `openai` (for embeddings + chat completions)
- `pypdf` **or** `pdfplumber` (choose one) for text extraction
- Vector store: **Pinecone** (recommended) via `pinecone-client`; you may alternatively use **FAISS** or a simple in‑memory index for development
- `numpy`, `re`, `math`, `hashlib` (as needed)

> You may import additional libraries if helpful. Use your judgment.

**Environment variables expected (for grading):**
- `OPENAI_API_KEY`
- If using Pinecone: `PINECONE_API_KEY`, `PINECONE_ENV` (or project config), and an index name you create in notebook (e.g., `pdf-mini-app`).



---

## Function Requirements
You'll have to implment the following functionality down below

### 1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> list[str]`

**Inputs**
- `pdf_path: str` — path to a readable PDF file (e.g., `data/sample.pdf`)
- `chunk_size: int` (default 800) — approximate number of characters per chunk
- `overlap: int` (default 120) — number of overlapping characters between adjacent chunks

**Output**
- `List[str]` — ordered, non-empty chunks of plain text. No chunk should exceed `chunk_size + 20` characters (allowing minor boundary variance).

**Behavioral constraints**
- Strip excessive whitespace and normalize newlines.
- Deterministic chunk boundaries given the same inputs.
- Apply overlap between chunks (except the first).

**Recommended approach**
1. Extract all PDF text into a single string using `pypdf` **or** `pdfplumber`.
2. Normalize whitespace.
3. Greedy chunk by character count with the requested overlap.



---

### 2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`

**Purpose**  
Create embeddings with **OpenAI** and store them in a vector index (Pinecone preferred).

**Inputs**
- `chunks: List[str]` — the output of `parse_pdf_to_chunks`
- `use_pinecone: bool` — if `True`, upsert to Pinecone; if `False`, store locally (FAISS or simple in‑memory array search)
- `model: str` — OpenAI embedding model name (e.g., `"text-embedding-3-small"`)

**Output**
- `dict` with fields (minimum):
  - `"chunks"`: `List[str]`
  - `"index_backend"`: `"pinecone"` or `"local"`
  - `"index_ref"`: backend-specific handle (e.g., Pinecone index name / namespace, or local matrix + ids)
  - `"embed_model"`: embedding model used
  - `"dim"`: embedding dimensionality

**Notes**
- When `use_pinecone=True`: create (if needed) and upsert vectors with stable IDs (e.g., `doc-0`, `doc-1`, …). Keep a mapping from ID → original chunk text in the dict.
- When `use_pinecone=False`: store embeddings in memory (e.g., NumPy matrix) along with their chunk IDs for cosine‑similarity search.



---

### 3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Purpose**  
Retrieve relevant chunks and generate an answer using an OpenAI chat model.

**Inputs**
- `question: str`
- `index_obj: dict` — the output from `embed_and_index`
- `top_k: int` — number of chunks to retrieve
- `gpt_model: str` — OpenAI chat model (e.g., `"gpt-4o-mini"`)

**Output**
- `str` — the final answer.

**Required behavior**
- Embed the **question** using the **same embedding model** used to build the index.
- Retrieve **top_k** chunks by cosine similarity (Pinecone query or local similarity search).
- Construct a concise prompt:
  - System: “You are a helpful assistant. Use ONLY the provided context to answer. If not answerable from context, respond: ‘I don’t know based on the document.’”
  - User: include the `question` and a `Context:` section concatenating the retrieved chunks (with simple separators).
- Call the OpenAI chat completion endpoint and **return the assistant’s message string**.



In [18]:
# This cell may be run by the grader
import os, re, math, hashlib
import numpy as np

### API Key Setup

You will be prompted to enter your keys securely below (they are **not** stored in the
notebook file, so it's safe to share/submit this notebook afterwards).

**How to get your OpenRouter API key** (required):
1. Go to [openrouter.ai](https://openrouter.ai) and sign up / log in.
2. Click your profile icon → **Keys** (or go directly to
   [openrouter.ai/keys](https://openrouter.ai/keys)).
3. Click **Create Key**, give it a name, and copy the key (starts with `sk-or-...`).
4. Add some credits to your account if required — OpenRouter is pay-as-you-go and
   many models have free or very low-cost tiers.
5. Paste it in when the notebook asks — it will not be shown on screen as you type.

**How to get your Pinecone API key** (optional — only needed if you want to use
Pinecone instead of the local in-memory index):
1. Go to [pinecone.io](https://www.pinecone.io) and sign up / log in.
2. In the Pinecone console, go to **API Keys** in the left sidebar.
3. Copy the **default** key (or click **Create API Key** to make a new one).
4. Paste it in when prompted. If you leave it blank, the notebook will
   automatically fall back to the local NumPy index — no Pinecone account needed.

> Never commit these keys to GitHub, paste them into screenshots, or hard-code
> them anywhere in this notebook.


In [19]:
import os
from getpass import getpass

# You will be prompted to paste each key below. Input is hidden as you type
# and is only kept in memory for this session -- it is never written to the
# notebook file, so this cell is safe to leave in your submission.

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key (required): ").strip()

pinecone_key = getpass(
    "Enter your Pinecone API key (optional -- press Enter to skip and use the local index): "
).strip()
if pinecone_key:
    os.environ['PINECONE_API_KEY'] = pinecone_key


Enter your OpenRouter API key (required): ··········
Enter your Pinecone API key (optional -- press Enter to skip and use the local index): ··········


In [20]:
OPENAI_API_KEY = OPENROUTER_API_KEY
assert OPENAI_API_KEY, "Please set OPENAI_API_KEY"
USE_PINECONE = bool(os.getenv("PINECONE_API_KEY"))

<center><h1>--------------- Student Code ---------------</h1></center>

### B. Implement `parse_pdf_to_chunks`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [21]:
import re

def parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120):
    """
    Extract text from a PDF and split it into overlapping chunks.

    Steps:
    1. Extract raw text from every page of the PDF using pypdf.
    2. Normalize whitespace (collapse repeated spaces/newlines into single spaces).
    3. Greedily slice the normalized text into chunks of `chunk_size` characters,
       stepping forward by (chunk_size - overlap) each time so that consecutive
       chunks share `overlap` characters. This keeps the chunking deterministic.

    Parameters
    ----------
    pdf_path : str
        Path to the PDF file to parse.
    chunk_size : int
        Target/maximum number of characters per chunk.
    overlap : int
        Number of characters shared between consecutive chunks.

    Returns
    -------
    List[str]
        Ordered list of non-empty text chunks.
    """
    from pypdf import PdfReader

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    # 1. Extract raw text from all pages
    reader = PdfReader(pdf_path)
    raw_text_parts = []
    for page in reader.pages:
        text = page.extract_text() or ""
        raw_text_parts.append(text)
    raw_text = "\n".join(raw_text_parts)

    # 2. Normalize whitespace: collapse newlines/tabs/multiple spaces into single spaces
    normalized = re.sub(r"\s+", " ", raw_text).strip()

    if not normalized:
        return []

    # 3. Greedy chunking with overlap, deterministic given the same inputs
    chunks = []
    step = chunk_size - overlap
    start = 0
    text_len = len(normalized)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = normalized[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == text_len:
            break
        start += step

    return chunks


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `parse_pdf_to_chunks`.
 -  If the function is not found, you answer will be rejected by the system

### C. Implement `embed_and_index`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [22]:
def _get_openai_client():
    """Create an OpenAI-compatible client pointed at OpenRouter."""
    from openai import OpenAI
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENAI_API_KEY,
    )


def _embed_texts(texts, model):
    """Embed a list of texts using the OpenRouter embeddings endpoint."""
    client = _get_openai_client()
    response = client.embeddings.create(model=model, input=texts)
    # response.data preserves the same order as the input list
    return [item.embedding for item in response.data]


def embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small"):
    """
    Embed a list of text chunks (via OpenRouter) and store them in a vector index.

    Parameters
    ----------
    chunks : List[str]
        Output of parse_pdf_to_chunks.
    use_pinecone : bool
        If True AND a PINECONE_API_KEY is configured, upsert vectors to Pinecone.
        Otherwise, falls back automatically to a local in-memory index.
    model : str
        Embedding model name to request through OpenRouter,
        e.g. "openai/text-embedding-3-small".

    Returns
    -------
    dict
        {
          "chunks": List[str],
          "index_backend": "pinecone" | "local",
          "index_ref": backend-specific handle,
          "embed_model": str,
          "dim": int,
        }
    """
    if not chunks:
        raise ValueError("chunks must be a non-empty list")

    ids = [f"doc-{i}" for i in range(len(chunks))]
    embeddings = _embed_texts(chunks, model=model)
    dim = len(embeddings[0])

    can_use_pinecone = use_pinecone and bool(os.environ.get("PINECONE_API_KEY"))

    if can_use_pinecone:
        from pinecone import Pinecone, ServerlessSpec

        pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
        index_name = "pdf-mini-app"

        existing = [idx["name"] for idx in pc.list_indexes()]
        if index_name not in existing:
            pc.create_index(
                name=index_name,
                dimension=dim,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1"),
            )
        index = pc.Index(index_name)

        vectors = [
            {"id": ids[i], "values": embeddings[i], "metadata": {"chunk": chunks[i], "order": i}}
            for i in range(len(chunks))
        ]
        index.upsert(vectors=vectors, namespace="pdf-mini-app")

        return {
            "chunks": chunks,
            "index_backend": "pinecone",
            "index_ref": {"index_name": index_name, "namespace": "pdf-mini-app", "ids": ids},
            "embed_model": model,
            "dim": dim,
        }

    # Local fallback: NumPy matrix + cosine similarity search
    matrix = np.array(embeddings, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1e-10
    normalized_matrix = matrix / norms

    return {
        "chunks": chunks,
        "index_backend": "local",
        "index_ref": {"ids": ids, "matrix": matrix, "normalized_matrix": normalized_matrix},
        "embed_model": model,
        "dim": dim,
    }


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `embed_and_index`.
 -  If the function is not found, you answer will be rejected by the system

### D. Implement `answer_query`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [29]:
def _cosine_top_k(query_vec, matrix_normalized, k):
    """Return indices and scores of the k most similar rows to query_vec."""
    q = np.array(query_vec, dtype=np.float32)
    q_norm = np.linalg.norm(q)
    if q_norm == 0:
        q_norm = 1e-10
    q_normalized = q / q_norm
    scores = matrix_normalized @ q_normalized
    top_idx = np.argsort(-scores)[:k]
    return top_idx, scores[top_idx]


def answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini"):
    """
    Answer a natural-language question using retrieval-augmented generation.

    Steps
    -----
    1. Embed the question with the same embedding model used to build the index.
    2. Retrieve the top_k most similar chunks (Pinecone query, or local cosine
       similarity search).
    3. Build a context-bounded prompt instructing the model to answer only
       from the retrieved context.
    4. Call the OpenRouter chat completion endpoint and return the answer text.

    Parameters
    ----------
    question : str
    index_obj : dict
        Output of embed_and_index.
    top_k : int
    gpt_model : str
        Chat model name to request through OpenRouter, e.g. "openai/gpt-4o-mini".

    Returns
    -------
    str
        The model's answer, grounded in the retrieved context.
    """
    embed_model = index_obj["embed_model"]
    question_embedding = _embed_texts([question], model=embed_model)[0]

    backend = index_obj["index_backend"]

    if backend == "pinecone":
        from pinecone import Pinecone

        pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
        ref = index_obj["index_ref"]
        index = pc.Index(ref["index_name"])
        result = index.query(
            vector=question_embedding,
            top_k=top_k,
            namespace=ref["namespace"],
            include_metadata=True,
        )
        retrieved_chunks = [match["metadata"]["chunk"] for match in result["matches"]]

    elif backend == "local":
        ref = index_obj["index_ref"]
        top_idx, _scores = _cosine_top_k(question_embedding, ref["normalized_matrix"], top_k)
        retrieved_chunks = [index_obj["chunks"][i] for i in top_idx]

    else:
        raise ValueError(f"Unknown index_backend: {backend}")

    context_block = "\n\n---\n\n".join(retrieved_chunks)

    system_prompt = (
        "You are a helpful assistant. Use ONLY the provided context to answer. "
        "If not answerable from context, respond: 'I don\'t know based on the document.'"
    )
    user_prompt = f"Question: {question}\n\nContext:\n{context_block}"

    client = _get_openai_client()
    response = client.chat.completions.create(
        model=gpt_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=1000,
    )

    return response.choices[0].message.content.strip()


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `answer_query`.
 -  If the function is not found, you answer will be rejected by the system


### E. (Optional) Demo cell (ungraded)
Provide a small demo that:
- Loads `data/sample.pdf`
- Runs all three functions
- Prints an example answer



In [24]:
!pip install pypdf

In [25]:
!pip install -q pinecone

In [30]:
# ------------------------------------------------------------------
# Demo: run the full pipeline end-to-end on a sample PDF
# Update PDF_PATH to point at your own PDF before running this cell.
# ------------------------------------------------------------------
PDF_PATH = "/content/Reflection.pdf"  # <-- update this path
import pypdf
import pinecone

chunks = parse_pdf_to_chunks(PDF_PATH, chunk_size=800, overlap=120)
print(f"Extracted {len(chunks)} chunks from {PDF_PATH}")

index_obj = embed_and_index(
    chunks,
    use_pinecone=USE_PINECONE,
    model="openai/text-embedding-3-small",
)
print(f"Index backend: {index_obj['index_backend']}, embedding dim: {index_obj['dim']}")


Extracted 8 chunks from /content/Reflection.pdf
Index backend: pinecone, embedding dim: 1536


In [31]:
def vanilla_llm_answer(question, gpt_model="openai/gpt-4o-mini"):
    """Answer a question with NO retrieval -- the model relies only on its own
    pretrained knowledge. Used as the baseline to compare against RAG answers."""
    client = _get_openai_client()
    response = client.chat.completions.create(
        model=gpt_model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question},
        ],
        max_tokens=1000,
    )
    return response.choices[0].message.content.strip()


In [32]:
# ------------------------------------------------------------------
# Comparative playtest: ask 3 domain-specific questions about the PDF
# and compare the vanilla LLM answer vs. the RAG-augmented answer.
# Replace the placeholder questions with ones specific to your PDF.
# ------------------------------------------------------------------
questions = [
    "Which model used the fewest completion tokens on the creative writing task, and how many tokens did it use?",
    "What specific instruction-following violation did Claude Haiku 4.5 commit, and why does the reflection say it matters?",
    "According to the reflection, which two OpenRouter model slugs returned 404 errors during debugging, and what does the author recommend to prevent this in the future?",
]

results = []
for q in questions:
    vanilla = vanilla_llm_answer(q)
    rag = answer_query(q, index_obj, top_k=3, gpt_model="openai/gpt-4o-mini")
    results.append({"question": q, "vanilla": vanilla, "rag": rag})
    print("Q:", q)
    print("  Vanilla LLM:", vanilla)
    print("  RAG LLM:    ", rag)
    print()


Q: Which model used the fewest completion tokens on the creative writing task, and how many tokens did it use?
  Vanilla LLM: I'm sorry, but I do not have access to specific performance metrics or token counts for different models on creative writing tasks. My responses are based on training data up to October 2023 and don't include specific details about model comparisons or individual task performances. If you have a specific model in mind or a particular dataset, I might be able to help you with general information or guidance.
  RAG LLM:     Claude Haiku 4.5 used the fewest completion tokens on the creative writing task, with a total of 356 tokens.

Q: What specific instruction-following violation did Claude Haiku 4.5 commit, and why does the reflection say it matters?
  Vanilla LLM: Claude Haiku 4.5 committed a specific instruction-following violation by failing to adhere to the user prompts accurately or completely. This could refer to not following the specific guidelines or par

In [33]:
# Optional: display results as a simple table for easy copy-paste into your reflection
import pandas as pd

df = pd.DataFrame(results)
df


,question,vanilla,rag
0,Which model used the fewest completion tokens ...,"I'm sorry, but I do not have access to specifi...",Claude Haiku 4.5 used the fewest completion to...
1,What specific instruction-following violation ...,Claude Haiku 4.5 committed a specific instruct...,Claude Haiku 4.5 violated the explicit instruc...
2,"According to the reflection, which two OpenRou...",The two OpenRouter model slugs that returned 4...,The two OpenRouter model slugs that returned 4...


### Reflection (500–700 words)

*Replace this text with your own written reflection. Suggested structure below.*

**1. How did RAG improve the answers compared with a vanilla LLM?**

Write 1–2 paragraphs comparing your `vanilla` vs `rag` answers from the table above.
Note any hallucinations in the vanilla answers, and where the RAG answers were more
factually grounded, more specific, or explicitly said "I don't know" instead of guessing.

**2. What role did chunking play in the quality of retrieval and answers?**

Discuss how `chunk_size` and `overlap` affected which passages got retrieved.
If you experimented with different chunk sizes, describe what changed.

**3. How did embeddings and vector search contribute to accuracy and relevance?**

Explain, in your own words, how cosine similarity over embedding vectors let the
system find semantically relevant chunks even when the question didn't use the
exact same wording as the PDF.

**4. What trade-offs did you observe (speed, cost, implementation complexity)?**

Consider: extra API calls for embeddings, added latency from retrieval, complexity
of maintaining an index vs. just calling the LLM directly, and any accuracy/cost
trade-offs between local (in-memory) vs. Pinecone-backed retrieval.

**5. How would you extend the system (e.g., caching/CAG, hybrid RAG+KAG)?**

Ideas to consider: caching embeddings so you don't re-embed the same PDF twice,
caching frequent question/answer pairs (CAG), combining RAG with a structured
knowledge graph (KAG) for facts that benefit from explicit relationships, or
adding a re-ranking step after retrieval.


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

Reflection (500–700 words)

1. How did RAG improve the answers compared with a vanilla LLM?

The gap was largest on the third question, asking which two OpenRouter model slugs returned 404 errors. The vanilla LLM confidently answered gpt-3.5-turbo and gpt-4 — a clean hallucination, since those aren't even OpenRouter-style provider-prefixed slugs, let alone the ones mentioned in my source document (anthropic/claude-3.5-haiku and google/gemini-2.0-flash-001). The RAG-augmented answer retrieved the correct passage and returned the exact slugs verbatim. On question 2 (Claude Haiku 4.5's instruction-following violation), the vanilla model gave a vague, generic answer about "failing to adhere to user prompts" with no specifics, while the RAG answer named the actual violation — wrapping JSON output in a markdown code fence despite an explicit instruction not to. Question 1 was more interesting: the RAG pipeline correctly grounded its answer in real numbers from the document (356 tokens, tied to Claude Haiku 4.5) but attached that figure to the wrong task — 356 was Claude Haiku's token count on the analytical reasoning task, not the creative writing task the question asked about (where Gemini 2.5 Flash was actually the leanest, at 433 tokens). This shows RAG reduces outright fabrication but doesn't eliminate retrieval/generation errors — it can still surface a real fact from the wrong context.

2. What role did chunking play in the quality of retrieval and answers?

With chunk_size=800 and overlap=120, my source document was split into only 8 chunks. Because both the creative-writing and analytical-reasoning sections discuss token counts in similar phrasing ("fewest tokens," "used X tokens"), a chunk boundary likely placed the analytical-reasoning token stats in — or adjacent to — the same or a highly similar-scoring chunk as the creative-writing discussion, which plausibly explains the mixed-up answer in question 1. This suggests that for documents with parallel, repetitive structure across sections, either smaller chunks with clearer section boundaries, or metadata tagging chunks by section/task, would improve retrieval precision.

3. How did embeddings and vector search contribute to accuracy and relevance?

Cosine similarity search successfully pulled directly relevant chunks for questions 2 and 3, even though the questions didn't quote the document's exact wording — asking "what violation did it commit" still retrieved the fence-wrapping passage. That's the main value of embeddings over keyword search: semantic closeness, not literal string matching. But question 1 shows the limits — semantic similarity between two sections both about "token counts" and "fewest tokens" isn't enough to guarantee the correct section is retrieved when multiple sections share vocabulary and structure.

4. What trade-offs did you observe (speed, cost, implementation complexity)?

Using Pinecone (embedding dimension 1536, pdf-mini-app index) added real setup overhead — index creation, upsert calls, metadata management — compared to a local NumPy array. For a one-off 8-chunk document, that complexity wasn't obviously worth it over the in-memory fallback; Pinecone's advantage would show up at larger scale or with persistence needs across sessions. Each RAG query also cost an extra embedding call plus the added retrieval step versus vanilla's single chat call, adding latency and API cost per question.

5. How would you extend the system (e.g., caching/CAG, hybrid RAG+KAG)?

Given the section-confusion error in question 1, I'd extend this system with section-aware chunking or a metadata filter (e.g., tagging each chunk with the task/heading it came from) so retrieval can disambiguate between structurally similar sections. I'd also add a lightweight re-ranking step after retrieval to catch cases where a high-similarity chunk is semantically close but topically wrong. For repeated grading runs, caching embeddings for the same PDF (avoiding re-embedding identical chunks) would cut cost, and a hybrid approach that pairs RAG with a small structured index of key facts (e.g., task → model → token count) would sidestep exactly this kind of table-like data getting scrambled by prose-chunk retrieval.

---

## Hidden Tests (run by grader)

Hidden tests will verify that:

- `parse_pdf_to_chunks` applies overlap correctly, is deterministic, and honors size bounds.
- `embed_and_index` calls OpenAI embeddings and builds a valid index (Pinecone or local), with stable IDs and correct dimensionality/shape.
- Retrieval uses cosine similarity and returns the expected top‑k order on crafted inputs.
- `answer_query` embeds the question with the same model used for chunks, constructs a context‑bounded prompt, calls OpenAI chat completions, and returns a clean string.
- No hard‑coded secrets; keys must be read from environment.
- Notebook runs top‑to‑bottom reproducibly when keys are present.



---

## Rubric (100 pts)

- **PDF parsing & chunking (30 pts)**
  - Correct extraction and normalization (10)
  - Deterministic overlapping chunks within size bounds (15)
  - Edge cases (empty pages, weird whitespace) (5)

- **Embeddings & indexing (35 pts)**
  - Correct OpenAI embedding usage and shapes (15)
  - Proper index structure + metadata (10)
  - Stable IDs and retrievability (10)

- **Query flow & answer (25 pts)**
  - Correct cosine similarity retrieval and stable tie‑break (10)
  - Prompt enforces context‑only answering; sensible output (15)

- **Code quality & constraints (10 pts)**
  - Clean, readable, documented code (5)
  - Secure key handling; graceful error handling (5)



---

## (Optional) Pinecone/OpenAI Notes

- For Pinecone: create an index (e.g., 384 or 1536 dim depending on embedding model) and upsert vectors with metadata (`{"chunk": "...", "order": i}`).
- For local dev without Pinecone: store embeddings in a NumPy matrix and do cosine similarity. Grading is OK with either backend as long as OpenAI embeddings + chat are used.
- Keep your mapping from vector IDs to original chunk text; you’ll need it for context construction.



---

## Submission

Submit your completed notebook. Ensure it runs top‑to‑bottom without modification in a clean environment **with valid API keys set**. The grader will run hidden tests using OpenAI for embeddings and chat.


---

<h2><center> Completing your test </center></h2>

- Once you have completed your test and answered all the questions:
  
- You can open the guide using the button located on the far right of your screen.

- The button will look like this image:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/End_guide_button.png?raw=1)

- Clicking on this button will allow you to mark the test as complete using the button shown below:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/completion_button.png?raw=1)

- Please click on **Mark as Completed** to end and submit your test.

- Thank you!
